# Faster R-CNN 학습 (PCB AOI 부품 탐지)

같은 데이터셋으로 학습한 YOLO, RTMDet과 비교하기 위한 Faster R-CNN 학습 노트북입니다.
평가는 `test.ipynb`에서 합니다.

**준비 (Colab)**
1. 런타임 → 런타임 유형 변경 → **T4 GPU**
2. 드라이브 `MyDrive/pcb-aoi/faster_rcnn/`에 `config.py`, `common.py`, 이 노트북을 올립니다.
3. 데이터셋 폴더 `MyDrive/Circuit testing 2 -test.v1i.coco/` (train / valid / test)가 있어야 합니다.

**로컬**에서는 `inference/dataset/`에 데이터셋을 두고 이 폴더에서 노트북을 열면 됩니다. 경로는 `config.py`가 자동으로 고릅니다.

**현재 가중치**: `best_fasterrcnn_20260923_1007.pth` (이 노트북, seed 42로 학습). 다시 학습하면 `best_fasterrcnn_<날짜_시각>.pth`로 새 파일이 만들어지므로, 평가에 쓰려면 `config.WEIGHTS_NAME`을 바꿔야 합니다.

## 1. 환경 설정

In [ ]:
# Colab이면 드라이브를 연결하고 config.py·common.py가 있는 드라이브 폴더로 이동한다.
# 로컬이면 이 노트북과 같은 폴더(faster_rcnn/)의 파일을 그대로 쓴다.
import os, sys, subprocess, importlib

PROJECT_DIR = "/content/drive/MyDrive/pcb-aoi/faster_rcnn"   # config.py, common.py를 올려 둔 드라이브 폴더

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir(PROJECT_DIR)
    if PROJECT_DIR not in sys.path:
        sys.path.insert(0, PROJECT_DIR)

import config, common
importlib.reload(config); importlib.reload(common)   # config.py를 고친 뒤 이 셀만 다시 실행해도 반영됨

for k, v in config.summary().items():
    print(f"{k:13s} {v}")

In [ ]:
import torch, torchvision
print("torch:", torch.__version__, "/ torchvision:", torchvision.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU 없음: 런타임 → 런타임 유형 변경 → T4 GPU로 바꾸세요.")

## 2. 데이터 준비

Colab에서는 드라이브에서 바로 읽으면 느리므로 `/content/data`로 복사해서 씁니다.
학습 json의 클래스 순서가 `config.CLASSES`와 같은지 먼저 확인합니다. 다르면 여기서 멈춥니다.

In [ ]:
import shutil
from pathlib import Path

if config.IN_COLAB:
    data_dir = Path("/content/data")
    if not data_dir.exists():
        shutil.copytree(config.DATA_ROOT, data_dir)
else:
    data_dir = config.DATA_ROOT
print("데이터 경로:", data_dir, sorted(os.listdir(data_dir)))

common.check_train_classes(data_dir / "train" / common.ANN_FILE, config.CLASSES)

train_ds = common.CocoDetectionDataset(data_dir / "train", config.CLASSES)
valid_ds = common.CocoDetectionDataset(data_dir / "valid", config.CLASSES, verbose=False)
print("\ntrain:", len(train_ds), "/ valid:", len(valid_ds))

## 3. 모델

COCO로 사전학습된 `fasterrcnn_resnet50_fpn`(v1)을 불러와 마지막 분류기만 우리 클래스 수(8 + 배경)로 바꿉니다 (전이학습).

In [ ]:
common.set_seed(config.SEED)
model = common.build_model(config.NUM_CLASSES, pretrained=True).to(device)
print("모델:", config.MODEL_NAME, "/ 출력 클래스 수:", config.NUM_CLASSES, "(배경 포함)")

## 4. 학습

- SGD + CosineAnnealing, 설정값은 `config.py`
- 매 에폭 검증 손실을 재고, 가장 낮을 때의 가중치를 저장합니다.
- 가중치는 `OUTPUT_DIR/best_fasterrcnn_<날짜_시각>.pth`로 저장되어 기존 파일을 덮어쓰지 않습니다.
- 에폭별 손실은 `results/train_log_<날짜_시각>.csv`로 남습니다.

In [ ]:
import csv, datetime, math, time
from torch.utils.data import DataLoader

g = torch.Generator().manual_seed(config.SEED)
train_loader = DataLoader(train_ds, batch_size=config.BATCH_SIZE, shuffle=True, generator=g,
                          collate_fn=common.collate_fn, num_workers=config.NUM_WORKERS)
valid_loader = DataLoader(valid_ds, batch_size=1, shuffle=False,
                          collate_fn=common.collate_fn, num_workers=config.NUM_WORKERS)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=config.LR, momentum=config.MOMENTUM,
                            weight_decay=config.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.EPOCHS)


def run_epoch(loader, train):
    # torchvision 검출 모델은 train 모드에서만 손실을 돌려준다.
    # 검증은 train 모드 + 기울기 끔으로 계산한다 (백본은 FrozenBatchNorm이라 통계가 바뀌지 않음).
    model.train()
    total = 0.0
    with torch.set_grad_enabled(train):
        for images, targets in loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss = sum(model(images, targets).values())
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total += loss.item()
    return total / len(loader)


stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
weights_path = config.OUTPUT_DIR / f"best_fasterrcnn_{stamp}.pth"

log, best, best_epoch = [], math.inf, 0
for epoch in range(1, config.EPOCHS + 1):
    t0 = time.time()
    lr = optimizer.param_groups[0]["lr"]
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(valid_loader, train=False)
    scheduler.step()

    saved = va < best
    if saved:
        best, best_epoch = va, epoch
        torch.save(model.state_dict(), weights_path)
    sec = time.time() - t0
    log.append({"epoch": epoch, "lr": lr, "train_loss": tr, "valid_loss": va,
                "seconds": round(sec, 1), "saved": int(saved)})
    print(f"epoch {epoch:3d}  lr {lr:.5f}  train {tr:.4f}  valid {va:.4f}  ({sec:.0f}s)"
          + ("  <- 저장" if saved else ""))

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
log_path = config.RESULTS_DIR / f"train_log_{stamp}.csv"
with open(log_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(log[0]))
    w.writeheader()
    w.writerows(log)
print("\n학습 로그:", log_path)

## 5. 학습 요약 저장

In [ ]:
summary = {
    "weights": weights_path.name,
    "model": config.MODEL_NAME,
    "best_epoch": best_epoch,
    "best_valid_loss": best,
    "epochs": config.EPOCHS,
    "batch_size": config.BATCH_SIZE,
    "lr": config.LR,
    "momentum": config.MOMENTUM,
    "weight_decay": config.WEIGHT_DECAY,
    "seed": config.SEED,
    "augmentation": "none",
    "train_images": len(train_ds),
    "valid_images": len(valid_ds),
    "device": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
}
common.save_json(summary, config.RESULTS_DIR / f"train_summary_{stamp}.json")
print(f"\n가장 좋은 에폭: {best_epoch} (valid {best:.4f})")
print(f"이 가중치로 평가하려면 config.py의 WEIGHTS_NAME을 {weights_path.name!r}로 바꾸세요.")

## 6. requirements.txt 기록

지금 환경에 설치된 실제 버전을 `requirements.txt`로 남깁니다. 추측한 버전을 적지 않기 위한 셀입니다.

In [ ]:
common.write_requirements(config.HERE / "requirements.txt")